In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

from chlforecast import config
from chlforecast import features as F

BASE = config.BASE

data = xr.open_dataset(config.PROCESSED / "malta_daily_observations.nc")

# Quality control (shared rule in chlforecast): chlorophyll above
# config.MAX_PLAUSIBLE_CHL is treated as missing. Applied before features and
# targets are built, so it affects both.
n_rejected = int((data["chl"] > config.MAX_PLAUSIBLE_CHL).sum().item())
chl = F.quality_control(data["chl"].transpose("time", "latitude", "longitude").values)
data["chl"] = (("time", "latitude", "longitude"), chl)
print(f"CHL values above {config.MAX_PLAUSIBLE_CHL} mg/m³ set to missing: {n_rejected:,}")

display(data)


CHL values above 10.0 mg/m³ set to missing: 246


<xarray.Dataset> Size: 5GB
Dimensions:         (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time            (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude        (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude       (longitude) float32 404B 13.91 13.92 13.93 ... 15.18 15.19
Data variables: (12/17)
    chl             (time, latitude, longitude) float32 353MB 0.08791 ... nan
    sst_c           (time, latitude, longitude) float32 353MB ...
    wind_u_asc      (time, latitude, longitude) float32 353MB ...
    wind_v_asc      (time, latitude, longitude) float32 353MB ...
    wind_speed_asc  (time, latitude, longitude) float32 353MB ...
    wind_time_asc   (time, latitude, longitude) datetime64[ns] 707MB ...
    ...              ...
    wind_qc_des     (time, latitude, longitude) float32 353MB ...
    chl_valid       (time, latitude, longitude) uint8 88MB ...
    sst_valid       (time, latitude, longitude) uint8 88MB ...
    wind_asc_valid  (time, latitude, longitude) uint8 88MB ...
    wind_des_valid  (time, latitude, longitude) uint8 88MB ...
    wind_any_valid  (time, latitude, longitude) uint8 88MB ...
Attributes:
    description:           Daily L3 CHL, SST and ASCAT wind on the CHL grid
    spatial_mapping:       Nearest source cell within half-cell tolerance
    time_definition:       UTC daily labels; wind acquisition timestamps reta...
    temporal_gap_filling:  None
    source_files:          {"chl": "data/chl_l3_l4_comparison/chl_L3_2008-01-...

In [2]:
FORECAST_ISSUE_TIME_UTC = config.FORECAST_ISSUE_TIME_UTC
FORECAST_HORIZONS = config.FORECAST_HORIZONS
CHL_AVAILABILITY_LAG_DAYS = config.CHL_LAG_DAYS

dates = pd.DatetimeIndex(data.time.values)

assert dates.is_unique, "Duplicate dates found."
assert dates.equals(
    pd.date_range(dates.min(), dates.max(), freq="D")
), "Expected consecutive daily dates at midnight."

timeline = pd.DataFrame(index=dates)
timeline.index.name = "issue_date"

timeline["issue_time_utc"] = dates + pd.Timedelta(hours=20, minutes=30)
timeline["latest_chl_date"] = (
    dates - pd.Timedelta(days=CHL_AVAILABILITY_LAG_DAYS)
)

for horizon in FORECAST_HORIZONS:
    timeline[f"target_date_plus_{horizon}"] = (
        dates + pd.Timedelta(days=horizon)
    )

display(timeline.head(3))

,issue_time_utc,latest_chl_date,target_date_plus_1,target_date_plus_4,target_date_plus_7
issue_date,,,,,
2008-01-01,2008-01-01 20:30:00,2007-12-31,2008-01-02,2008-01-05,2008-01-08
2008-01-02,2008-01-02 20:30:00,2008-01-01,2008-01-03,2008-01-06,2008-01-09
2008-01-03,2008-01-03 20:30:00,2008-01-02,2008-01-04,2008-01-07,2008-01-10


In [3]:
FEATURE_LAGS = {
    "chl": config.CHL_LAG_DAYS,
    "sst_c": config.SST_LAG_DAYS,
    "wind_u_asc": config.WIND_LAG_DAYS,
    "wind_v_asc": config.WIND_LAG_DAYS,
    "wind_speed_asc": config.WIND_LAG_DAYS,
    "wind_u_des": config.WIND_LAG_DAYS,
    "wind_v_des": config.WIND_LAG_DAYS,
    "wind_speed_des": config.WIND_LAG_DAYS,
}


available = xr.Dataset({
    variable: data[variable].shift(time=lag)
    for variable, lag in FEATURE_LAGS.items()
})

for variable, lag in FEATURE_LAGS.items():
    available[variable].attrs["source_date_lag_days"] = lag

available.attrs["time_meaning"] = "Forecast issue date"
available.attrs["forecast_issue_time_utc"] = FORECAST_ISSUE_TIME_UTC
available.attrs["availability_basis"] = "Nominal NRT delivery schedules"

display(available)

<xarray.Dataset> Size: 3GB
Dimensions:         (latitude: 133, longitude: 101, time: 6575)
Coordinates:
  * latitude        (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude       (longitude) float32 404B 13.91 13.92 13.93 ... 15.18 15.19
  * time            (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
Data variables:
    chl             (time, latitude, longitude) float32 353MB nan nan ... nan
    sst_c           (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_u_asc      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_v_asc      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_speed_asc  (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_u_des      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_v_des      (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_speed_des  (time, latitude, longitude) float32 353MB nan nan ... nan
Attributes:
    time_meaning:             Forecast issue date
    forecast_issue_time_utc:  20:30
    availability_basis:       Nominal NRT delivery schedules

In [4]:
targets = xr.Dataset({
    f"chl_target_{horizon}d": data["chl"].shift(time=-horizon)
    for horizon in FORECAST_HORIZONS
})

for horizon in FORECAST_HORIZONS:
    targets[f"chl_target_{horizon}d"].attrs.update({
        "forecast_horizon_days": horizon,
        "units": "milligram m^-3",
        "description": (
            f"Observed L3 CHL {horizon} days after the forecast issue date"
        ),
    })

targets.attrs["time_meaning"] = "Forecast issue date"

display(targets)

<xarray.Dataset> Size: 1GB
Dimensions:        (latitude: 133, longitude: 101, time: 6575)
Coordinates:
  * latitude       (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude      (longitude) float32 404B 13.91 13.92 13.93 ... 15.18 15.19
  * time           (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
Data variables:
    chl_target_1d  (time, latitude, longitude) float32 353MB nan nan ... nan nan
    chl_target_4d  (time, latitude, longitude) float32 353MB nan nan ... nan nan
    chl_target_7d  (time, latitude, longitude) float32 353MB nan nan ... nan nan
Attributes:
    time_meaning:  Forecast issue date

In [5]:
SPLIT_PERIODS = config.SPLIT_PERIODS


latest_target_dates = dates + pd.Timedelta(
    days=max(FORECAST_HORIZONS)
)

split_masks = {}
rows = []

for name, (start, end) in SPLIT_PERIODS.items():
    mask = (
        (dates >= pd.Timestamp(start))
        & (latest_target_dates <= pd.Timestamp(end))
    )

    split_masks[name] = xr.DataArray(
        mask, dims="time", coords={"time": data.time}
    )

    selected = dates[mask]
    rows.append({
        "Split": name,
        "First issue date": selected.min().date(),
        "Last issue date": selected.max().date(),
        "Issue dates": len(selected),
    })

display(pd.DataFrame(rows).set_index("Split"))

,First issue date,Last issue date,Issue dates
Split,,,
train,2017-01-01,2023-12-24,2549
validation,2024-01-01,2024-12-24,359
test,2025-01-01,2025-12-24,358


In [6]:
SPATIAL_DIMS = ("latitude", "longitude")
GRID = ("time", *SPATIAL_DIMS)
grid_coords = {"time": data.time, "latitude": data.latitude, "longitude": data.longitude}

# Pixels observed fewer than config.MIN_TRAINING_OBSERVATIONS times in the
# training years are mostly coastal retrievals with unreliable values.
mask_values, training_counts = F.model_mask(chl, dates)

model_mask = xr.DataArray(
    mask_values, dims=SPATIAL_DIMS,
    coords={"latitude": data.latitude, "longitude": data.longitude},
    name="model_mask",
)
model_mask.attrs["description"] = (
    f"Pixels with at least {config.MIN_TRAINING_OBSERVATIONS} valid CHL "
    "observations during 2017–2023"
)

included_pixels = int(model_mask.sum().item())
rarely_observed = int(
    ((training_counts > 0) & (training_counts < config.MIN_TRAINING_OBSERVATIONS)).sum()
)

display(pd.Series({
    "Grid pixels": model_mask.size,
    "Included pixels": included_pixels,
    "Excluded: rarely observed": rarely_observed,
    "Included (%)": 100 * included_pixels / model_mask.size,
}).to_frame("Value").style.format({
    "Value": "{:,.2f}"
}))


,Value
Grid pixels,"13,433.00"
Included pixels,"12,190.00"
Excluded: rarely observed,172.00
Included (%),90.75


In [7]:
season_sin, season_cos = F.season_features(dates)

features = xr.Dataset(
    {
        "season_sin": ("time", season_sin),
        "season_cos": ("time", season_cos),
    },
    coords={"time": data.time},
)

features.attrs["time_meaning"] = "Forecast issue date"
features.attrs["forecast_issue_time_utc"] = FORECAST_ISSUE_TIME_UTC

display(features.isel(time=slice(0, 5)).to_dataframe())


,season_sin,season_cos
time,,
2008-01-01,0.000000,1.000000
2008-01-02,0.017166,0.999853
2008-01-03,0.034328,0.999411
2008-01-04,0.051479,0.998674
2008-01-05,0.068615,0.997643


In [8]:
# Latest observation, its age, and the 7-day mean and count (shared recipe).
chl_history = F.chl_history_features(chl, mask_values)

features["chl_latest"] = xr.DataArray(chl_history["chl_latest"], dims=GRID, coords=grid_coords)
features["chl_age_days"] = xr.DataArray(chl_history["chl_age_days"], dims=GRID, coords=grid_coords)

features["chl_latest"].attrs["units"] = "milligram m^-3"
features["chl_age_days"].attrs["units"] = "days"

display(features[["chl_latest", "chl_age_days"]])


<xarray.Dataset> Size: 707MB
Dimensions:       (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time          (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude      (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude     (longitude) float32 404B 13.91 13.92 13.93 ... 15.18 15.19
Data variables:
    chl_latest    (time, latitude, longitude) float32 353MB nan nan ... 0.07673
    chl_age_days  (time, latitude, longitude) float32 353MB nan nan ... 19.0 3.0
Attributes:
    time_meaning:             Forecast issue date
    forecast_issue_time_utc:  20:30

In [9]:
sst_inputs = F.sst_features(data["sst_c"].transpose(*GRID).values, mask_values)

features["sst_c"] = xr.DataArray(sst_inputs["sst_c"], dims=GRID, coords=grid_coords)
features["sst_observed"] = xr.DataArray(sst_inputs["sst_observed"], dims=GRID, coords=grid_coords)

features["sst_c"].attrs["units"] = "degree_Celsius"
features["sst_observed"].attrs["description"] = (
    "1 = eligible SST observation present; 0 = missing"
)

display(features[["sst_c", "sst_observed"]])


<xarray.Dataset> Size: 442MB
Dimensions:       (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time          (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude      (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude     (longitude) float32 404B 13.91 13.92 13.93 ... 15.18 15.19
Data variables:
    sst_c         (time, latitude, longitude) float32 353MB nan nan ... nan nan
    sst_observed  (time, latitude, longitude) uint8 88MB 0 0 0 0 0 ... 0 0 0 0 0
Attributes:
    time_meaning:             Forecast issue date
    forecast_issue_time_utc:  20:30

In [10]:
wind_feature_names = []

for pass_name in ["asc", "des"]:
    variables = [
        f"wind_u_{pass_name}",
        f"wind_v_{pass_name}",
        f"wind_speed_{pass_name}",
    ]

    valid = (
        np.isfinite(available[variables])
        .to_array(dim="component")
        .all(dim="component")
        & model_mask
    )

    for variable in variables:
        features[variable] = available[variable].where(valid)
        features[variable].attrs["units"] = "m s-1"

    flag = f"wind_{pass_name}_observed"
    features[flag] = valid.astype("uint8")
    features[flag].attrs["description"] = (
        "1 = all three eligible wind measurements present; 0 = missing"
    )

    wind_feature_names.extend(variables + [flag])

display(features[wind_feature_names])

<xarray.Dataset> Size: 2GB
Dimensions:            (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time               (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude           (latitude) float32 532B 35.55 35.56 35.57 ... 36.88 36.89
  * longitude          (longitude) float32 404B 13.91 13.92 ... 15.18 15.19
Data variables:
    wind_u_asc         (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_v_asc         (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_speed_asc     (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_asc_observed  (time, latitude, longitude) uint8 88MB 0 0 0 0 ... 0 0 0
    wind_u_des         (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_v_des         (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_speed_des     (time, latitude, longitude) float32 353MB nan nan ... nan
    wind_des_observed  (time, latitude, longitude) uint8 88MB 0 0 0 0 ... 0 0 0
Attributes:
    time_meaning:             Forecast issue date
    forecast_issue_time_utc:  20:30

In [11]:
issue_hour, issue_minute = map(
    int, FORECAST_ISSUE_TIME_UTC.split(":")
)

issue_time_utc = (
    data.time
    + np.timedelta64(issue_hour, "h")
    + np.timedelta64(issue_minute, "m")
)

rows = []

for pass_name in ["asc", "des"]:
    acquisition_time = data[f"wind_time_{pass_name}"].shift(
        time=FEATURE_LAGS[f"wind_speed_{pass_name}"]
    )

    age_hours = (
        (issue_time_utc - acquisition_time) / np.timedelta64(1, "h")
    )

    variable = f"wind_age_hours_{pass_name}"
    features[variable] = age_hours.astype("float32").where(
        features[f"wind_{pass_name}_observed"] == 1
    )
    features[variable].attrs["units"] = "hours"

    assert not bool((features[variable] < 0).any().item())

    rows.append({
        "Pass": pass_name,
        "Minimum age (hours)": float(features[variable].min().item()),
        "Maximum age (hours)": float(features[variable].max().item()),
    })

display(pd.DataFrame(rows).set_index("Pass").round(2))

,Minimum age (hours),Maximum age (hours)
Pass,,
asc,23.68,25.58
des,34.85,36.71


In [12]:
features["chl_mean_7d"] = xr.DataArray(chl_history["chl_mean_7d"], dims=GRID, coords=grid_coords)
features["chl_mean_7d"].attrs.update({
    "units": "milligram m^-3",
    "description": "Mean of valid CHL observations in the latest eligible 7 days",
})

features["chl_observations_7d"] = xr.DataArray(chl_history["chl_observations_7d"], dims=GRID, coords=grid_coords)
features["chl_observations_7d"].attrs["description"] = (
    "Number of valid CHL observations supporting chl_mean_7d, from 0 to 7"
)

display(features[["chl_mean_7d", "chl_observations_7d"]])


<xarray.Dataset> Size: 442MB
Dimensions:              (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time                 (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude             (latitude) float32 532B 35.55 35.56 ... 36.88 36.89
  * longitude            (longitude) float32 404B 13.91 13.92 ... 15.18 15.19
Data variables:
    chl_mean_7d          (time, latitude, longitude) float32 353MB nan ... 0....
    chl_observations_7d  (time, latitude, longitude) uint8 88MB 0 0 0 ... 0 0 1
Attributes:
    time_meaning:             Forecast issue date
    forecast_issue_time_utc:  20:30

In [13]:
# Monthly climatology of log10 CHL, from the training years only (shared recipe).
monthly_log_clim = F.monthly_log_climatology(chl, dates, mask_values)

features["chl_clim_log"] = xr.DataArray(
    F.climatology_for_dates(monthly_log_clim, dates), dims=GRID, coords=grid_coords
)
features["chl_clim_log"].attrs.update({
    "units": "log10(milligram m^-3)",
    "description": (
        "Median log10 CHL for the issue month at this pixel, 2017–2023"
    ),
})

display(
    pd.Series(np.nanmean(monthly_log_clim, axis=(1, 2)), index=pd.RangeIndex(1, 13, name="month"))
    .rename("Mean log10 CHL").to_frame()
)


,Mean log10 CHL
month,
1,-0.894262
2,-0.922928
3,-1.029872
4,-1.167305
5,-1.271433
6,-1.306006
7,-1.347380
8,-1.373986
9,-1.364925


In [14]:
features["latitude_deg"] = data.latitude.astype("float32")
features["longitude_deg"] = data.longitude.astype("float32")

features["latitude_deg"].attrs["units"] = "degrees_north"
features["longitude_deg"].attrs["units"] = "degrees_east"

feature_summary = pd.DataFrame([
    {
        "Feature": name,
        "Dimensions": ", ".join(variable.dims),
        "Units": variable.attrs.get("units", ""),
        "Type": str(variable.dtype),
    }
    for name, variable in features.data_vars.items()
]).set_index("Feature")

display(feature_summary)
print(f"Total features: {len(features.data_vars)}")

,Dimensions,Units,Type
Feature,,,
season_sin,time,,float32
season_cos,time,,float32
chl_latest,"time, latitude, longitude",milligram m^-3,float32
chl_age_days,"time, latitude, longitude",days,float32
sst_c,"time, latitude, longitude",degree_Celsius,float32
sst_observed,"time, latitude, longitude",,uint8
wind_u_asc,"time, latitude, longitude",m s-1,float32
wind_v_asc,"time, latitude, longitude",m s-1,float32
wind_speed_asc,"time, latitude, longitude",m s-1,float32


Total features: 21


In [15]:
availability_checks = {
    "CHL history (%)": np.isfinite(features["chl_latest"]),
    "SST (%)": features["sst_observed"] == 1,
    "Either wind (%)": (
        (features["wind_asc_observed"] == 1)
        | (features["wind_des_observed"] == 1)
    ),
}

rows = []

for split_name, date_mask in split_masks.items():
    eligible_count = int(date_mask.sum().item()) * included_pixels

    for horizon in FORECAST_HORIZONS:
        valid_target = (
            np.isfinite(targets[f"chl_target_{horizon}d"])
            & model_mask
            & date_mask
        )
        count = int(valid_target.sum().item())

        row = {
            "Split": split_name,
            "Horizon": f"+{horizon}d",
            "Valid targets": count,
            "Target coverage (%)": 100 * count / eligible_count,
        }

        for label, condition in availability_checks.items():
            row[label] = (
                100 * int((valid_target & condition).sum().item()) / count
                if count else np.nan
            )

        rows.append(row)

sample_summary = pd.DataFrame(rows).set_index(["Split", "Horizon"])

display(sample_summary.style.format({
    column: "{:,.0f}" if column == "Valid targets" else "{:.1f}"
    for column in sample_summary.columns
}))

In [16]:
age_labels = ["1", "2", "3", "4–7", "8–14", "15–30", "31+"]
age_bins = [0.5, 1.5, 2.5, 3.5, 7.5, 14.5, 30.5, np.inf]

rows = []

for split_name, date_mask in split_masks.items():
    valid_target = (
        np.isfinite(targets["chl_target_1d"])
        & model_mask
        & date_mask
    )

    ages = features["chl_age_days"].values[valid_target.values]
    known = np.isfinite(ages)
    counts, _ = np.histogram(ages[known], bins=age_bins)

    row = {"Split": split_name}
    row.update({
        label: 100 * count / ages.size
        for label, count in zip(age_labels, counts)
    })
    row["No history"] = 100 * (~known).sum() / ages.size
    rows.append(row)

history_age_summary = pd.DataFrame(rows).set_index("Split")
history_age_summary.columns.name = "CHL acquisition age (days)"

display(history_age_summary.style.format("{:.1f}%"))

CHL acquisition age (days),1,2,3,4–7,8–14,15–30,31+,No history
Split,,,,,,,,
train,63.5%,17.1%,8.8%,9.2%,1.2%,0.1%,0.0%,0.0%
validation,63.0%,16.1%,8.7%,10.9%,1.1%,0.1%,0.0%,0.0%
test,68.8%,17.2%,7.4%,6.2%,0.4%,0.0%,0.0%,0.0%


In [17]:
import json

model_data = xr.merge(
    [features, targets],
    join="exact",
    compat="no_conflicts",
    combine_attrs="override",
)

model_data["model_mask"] = model_mask.astype("uint8")

for name, mask in split_masks.items():
    model_data[f"split_{name}"] = mask.astype("uint8")

model_data.attrs.update({
    "feature_names_json": json.dumps(list(features.data_vars)),
    "forecast_horizons_days": list(FORECAST_HORIZONS),
    "forecast_issue_time_utc": FORECAST_ISSUE_TIME_UTC,
    "availability_basis": (
        "Nominal NRT delivery schedules applied to MY observations"
    ),
    "target_source": "Observed L3 chlorophyll",
    "training_period": "2017–2023",
    "validation_period": "2024",
    "test_period": "2025",
})

display(model_data)

<xarray.Dataset> Size: 6GB
Dimensions:              (time: 6575, latitude: 133, longitude: 101)
Coordinates:
  * time                 (time) datetime64[ns] 53kB 2008-01-01 ... 2025-12-31
  * latitude             (latitude) float32 532B 35.55 35.56 ... 36.88 36.89
  * longitude            (longitude) float32 404B 13.91 13.92 ... 15.18 15.19
Data variables: (12/28)
    season_sin           (time) float32 26kB 0.0 0.01717 ... -0.03442 -0.01721
    season_cos           (time) float32 26kB 1.0 0.9999 0.9994 ... 0.9994 0.9999
    chl_latest           (time, latitude, longitude) float32 353MB nan ... 0....
    chl_age_days         (time, latitude, longitude) float32 353MB nan ... 3.0
    sst_c                (time, latitude, longitude) float32 353MB nan ... nan
    sst_observed         (time, latitude, longitude) uint8 88MB 0 0 0 ... 0 0 0
    ...                   ...
    chl_target_4d        (time, latitude, longitude) float32 353MB nan ... nan
    chl_target_7d        (time, latitude, longitude) float32 353MB nan ... nan
    model_mask           (latitude, longitude) uint8 13kB 1 1 1 1 1 ... 0 1 1 1
    split_train          (time) uint8 7kB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    split_validation     (time) uint8 7kB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    split_test           (time) uint8 7kB 0 0 0 0 0 0 0 0 0 ... 1 0 0 0 0 0 0 0
Attributes:
    time_meaning:             Forecast issue date
    forecast_issue_time_utc:  20:30
    feature_names_json:       ["season_sin", "season_cos", "chl_latest", "chl...
    forecast_horizons_days:   [1, 4, 7]
    availability_basis:       Nominal NRT delivery schedules applied to MY ob...
    target_source:            Observed L3 chlorophyll
    training_period:          2017–2023
    validation_period:        2024
    test_period:              2025

In [18]:
output_path = BASE / "data/processed/malta_forecast_prepared.nc"
output_path.parent.mkdir(parents=True, exist_ok=True)

to_save = model_data.copy(deep=False)
to_save.attrs["source_observations_file"] = "malta_daily_observations.nc"

# Use fresh storage settings for the derived variables.
for name in to_save.variables:
    to_save[name].encoding = {}

encoding = {}

for name, variable in to_save.data_vars.items():
    options = {"zlib": True, "complevel": 4}

    if np.issubdtype(variable.dtype, np.floating):
        options.update(dtype="float32", _FillValue=np.nan)
    else:
        options.update(dtype="uint8", _FillValue=None)

    encoding[name] = options

encoding["time"] = {
    "dtype": "int64",
    "units": "days since 1970-01-01",
    "calendar": "proleptic_gregorian",
    "_FillValue": None,
}

to_save.to_netcdf(
    output_path,
    engine="netcdf4",
    encoding=encoding,
)

with xr.open_dataset(output_path) as saved:
    assert dict(saved.sizes) == dict(model_data.sizes)
    assert set(saved.data_vars) == set(model_data.data_vars)

print(f"Saved: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024**2:,.1f} MB")

Saved: /Users/alangatt/git/mcast.pages.chl_prediction/experiment_v2/data/processed/malta_forecast_prepared.nc
File size: 763.5 MB
